# Signisa — ASL Citizen holistic extraction (CPU, internet ON)

Attach the ASL Citizen Kaggle mirror (**kaggle.com/datasets/abd0kamel/asl-citizen**).
Extraction of ~83k clips does not fit one CPU session, so it is sharded: run this
notebook NUM_SHARDS times (separate sessions), once per SHARD_INDEX, and publish each
run's output. Attach every published output to `kaggle_prep.ipynb`, which merges them.

The partition is deterministic (sorted list, slice i::n), so re-running a shard after
an interrupt resumes where it stopped (completed npz files and logged failures are
skipped). The first cell is an integrity gate: a partial mirror hard-fails rather than
silently extracting a subset — fall back to the Microsoft download if it trips.

In [ ]:
# CONFIG — the only cell to edit
SHARD_INDEX = 0   # 0 .. NUM_SHARDS-1, one session each
NUM_SHARDS = 4
WORKERS = 4       # Kaggle CPU sessions have 4 cores

In [ ]:
!git clone -q https://github.com/dayan-battulga/signisa.git /kaggle/working/signisa-repo
%pip install -q "/kaggle/working/signisa-repo[live]"
!curl -sfL --create-dirs -o /kaggle/working/model/holistic_landmarker.task \
    https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task

In [ ]:
# Integrity gate: a partial mirror must hard-fail, never silently extract a subset
import shutil
from pathlib import Path

VIDEO_SUFFIXES = {".mp4", ".mov", ".mkv", ".webm", ".avi"}
MIN_VIDEOS = 80_000  # full ASL Citizen is ~83k clips

by_root = {}
for root in Path("/kaggle/input").iterdir():
    vids = [p for p in root.rglob("*") if p.suffix.lower() in VIDEO_SUFFIXES]
    if vids:
        by_root[root] = vids
assert by_root, "no video files under /kaggle/input — is the ASL Citizen mirror attached?"
DATASET_ROOT, videos = max(by_root.items(), key=lambda kv: len(kv[1]))

split_csvs = {p.stem: p for p in DATASET_ROOT.rglob("*.csv")
              if p.stem in ("train", "val", "test")}
print(f"{DATASET_ROOT}: {len(videos)} videos; split CSVs: "
      f"{ {k: str(v) for k, v in split_csvs.items()} }")
assert len(videos) >= MIN_VIDEOS, (
    f"only {len(videos)} videos (< {MIN_VIDEOS}) — partial mirror, fall back to the "
    "Microsoft ASL Citizen download instead of extracting a subset")
assert set(split_csvs) == {"train", "val", "test"}, (
    f"missing split CSVs (found {sorted(split_csvs)}) — partial mirror, use the "
    "Microsoft download")

# copy the split CSVs into the output so every shard is self-contained for prep
splits_out = Path("/kaggle/working/splits")
splits_out.mkdir(exist_ok=True)
for name, p in split_csvs.items():
    shutil.copy(p, splits_out / f"{name}.csv")

In [ ]:
# Extract this shard (resumable; prints clips/s + projected shard time by clip 200)
!python /kaggle/working/signisa-repo/scripts/extract_holistic.py \
    --videos-dir {DATASET_ROOT} \
    --out-dir /kaggle/working/extracted \
    --model /kaggle/working/model/holistic_landmarker.task \
    --workers {WORKERS} --shard-index {SHARD_INDEX} --num-shards {NUM_SHARDS}

In [ ]:
# Totals + shard manifest (kaggle_prep globs for shard_manifest.json)
import csv, json

out = Path("/kaggle/working/extracted")
npz = list(out.glob("*.npz"))
failures = out / "failures.csv"
n_failed = sum(1 for _ in csv.DictReader(failures.open())) if failures.exists() else 0
size_gb = sum(p.stat().st_size for p in npz) / 1e9
manifest = {"shard_index": SHARD_INDEX, "num_shards": NUM_SHARDS,
            "n_extracted": len(npz), "n_failed": n_failed,
            "n_shard_videos": len(videos[SHARD_INDEX::NUM_SHARDS]),
            "size_gb": round(size_gb, 2)}
json.dump(manifest, open("/kaggle/working/shard_manifest.json", "w"), indent=1)
print(manifest)
assert size_gb < 20, "over the notebook-output budget — raise NUM_SHARDS"
done = manifest["n_extracted"] + manifest["n_failed"]
if done < manifest["n_shard_videos"]:
    print(f"INCOMPLETE: {done}/{manifest['n_shard_videos']} — save a Version and "
          "re-run this notebook with the same SHARD_INDEX to resume")